# 21b · Fairness by group, adjusted for prompt (every attribute)

**Goal.** Notebook 21 gave raw false-flag rates by group and adjusted for prompt on only four comparisons, chosen after seeing the table. Here the same adjustment is applied to **every** attribute, with the same method, so nothing is singled out.

**Why prompts matter.** The prompts differ a lot in how often the system falsely flags a student (Summer projects 1.60%, Electoral college 0.11% at operating point A), and the groups are not spread evenly over prompts (almost every grade-11 essay is a Summer-projects essay). A raw gap between groups can therefore be a gap between prompts.

**Method (fixed here, before computing).**
1. *Raw:* flag rate per group at operating point A (flag if p_ens <= 0.00235), 95% Wilson interval.
2. *Prompt-adjusted:* the **Mantel-Haenszel risk difference**. Inside each prompt, take the flag-rate difference between a group and the reference group (the largest group); average these differences over prompts with weight n1*n0/(n1+n0) (prompts where one of the two groups is absent drop out). Interval: cluster bootstrap on `dup_cluster`, 1,000 resamples, seed 42.
3. *Prompt mix alone:* the rate a group would have if each of its essays were flagged at the overall rate of its prompt; observed minus this expected rate says how much of the raw rate is explained by which prompts the group wrote.
4. A prompt only counts toward a comparison if **both groups have at least 20 essays in it**. *This rule was added after the first run, which showed grade 9 and grade 11 sharing a single prompt in which grade 9 has 8 essays; an adjustment resting on 8 essays is not an adjustment.* A group with no usable prompt is reported as not estimable.
5. For grade, a second comparison, **grade 11 against grade 10**, is added after seeing the grade table (they share Summer projects with 1,527 and 209 essays).
6. All 14 comparisons against a reference group are reported (about 25 group rates in total). With 95% intervals about 1 in 20 comparisons would exclude zero by chance alone.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')
import numpy as np, pandas as pd
from src.data import load_splits
from src.evaluate import wilson_interval, ClusterSampler
from src.ensemble import PEnsemble
SEED, N_BOOT, MIN_STRATUM = 42, 1000, 20
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()
COLS = ['ell_status', 'gender', 'grade_level', 'race_ethnicity', 'economically_disadvantaged', 'student_disability_status']
dm = pd.read_csv('../data/external/persuade_demographics.csv', index_col='row_id')
h = data[data['label'] == 0].join(dm[COLS])
known = h[h['gender'].notna() | h['ell_status'].notna()].groupby('dup_cluster')[COLS].first()
miss = h['gender'].isna() & h['ell_status'].isna()
h.loc[miss, COLS] = known.reindex(h.loc[miss, 'dup_cluster']).to_numpy()

**Step 1 · the same evaluation set and scores as notebook 21** (test humans re-scored with the final ensemble; held-out-prompt humans taken from the final check, nothing re-scored).

In [2]:
vt = data[data['split'].isin(['val', 'test'])]
s42 = pd.read_csv('../results/15_scores.csv', index_col='row_id')['roberta']
f43 = pd.read_csv('../results/17_final/orig_s43_scores.csv', index_col='row_id')['logit']; f44 = pd.read_csv('../results/17_final/orig_s44_scores.csv', index_col='row_id')['logit']
vh = vt[(vt['split'] == 'val') & (vt['label'] == 0)].index
ens = PEnsemble(np.stack([s42.loc[vh], f43.loc[vh], f44.loc[vh]], axis=1), 'mean')
th = h[h['split'] == 'test']
test_part = pd.DataFrame({'p_ens': ens.p_ens(np.stack([s42.loc[th.index], f43.loc[th.index], f44.loc[th.index]], axis=1))}, index=th.index)
ho20 = pd.read_csv('../results/20_heldout_scores.csv', index_col='row_id')
hp = h[h['split'] == 'heldout_prompt']
ev = pd.concat([h.loc[test_part.index].join(test_part), h.loc[hp.index].join(ho20.loc[hp.index, ['p_ens']])])
ev['flag'] = ev['p_ens'] <= 0.00235
for c in COLS:
    ev[c] = ev[c].where(ev[c].notna(), 'unknown').astype(str).str.strip().replace({'': 'unknown', 'nan': 'unknown', 'None': 'unknown'})
ev['grade_level'] = ev['grade_level'].str.replace('.0', '', regex=False)
print(len(ev), 'humans;', int(ev['flag'].sum()), 'flagged;', ev['prompt_name'].nunique(), 'prompts')
print(ev.groupby('prompt_name')['flag'].agg(['size', 'sum', 'mean']).round(4).sort_values('mean', ascending=False))

6755 humans; 33 flagged; 15 prompts
                                       size  sum    mean
prompt_name                                             
Summer projects                        1750   28  0.0160
Seeking multiple opinions               192    1  0.0052
Car-free cities                         247    1  0.0040
Does the electoral college work?       2714    3  0.0011
Community service                       142    0  0.0000
Cell phones at school                   163    0  0.0000
"A Cowboy Who Rode the Waves"           134    0  0.0000
Driverless cars                         182    0  0.0000
Distance learning                       202    0  0.0000
Exploring Venus                         194    0  0.0000
Facial action coding system             228    0  0.0000
Mandatory extracurricular activities    169    0  0.0000
Grades for extracurricular activities   153    0  0.0000
Phones and driving                      128    0  0.0000
The Face on Mars                        157    0  0.

**Step 2 · the three quantities per group.** `mh_diff` is the Mantel-Haenszel risk difference; `expected_from_prompt` is the rate explained by prompt mix alone (step 3 of the method). The bootstrap redraws whole `dup_cluster` groups, so near-duplicate essays move together.

In [3]:
def wil(k, n):
    lo, hi = wilson_interval(int(k), int(n)); return f'{100 * k / n:.2f} [{100 * lo:.2f}, {100 * hi:.2f}]'

def stats(d, attr, g, ref, rows, pcode, P):
    # rows: indices into d (a bootstrap draw or all). Returns raw group rate, reference rate, MH diff, observed minus prompt-expected
    a = (d[attr].to_numpy()[rows] == g); b = (d[attr].to_numpy()[rows] == ref); f = d['flag'].to_numpy()[rows].astype(float); pc = pcode[rows]
    n1 = np.bincount(pc[a], minlength=P).astype(float); k1 = np.bincount(pc[a], weights=f[a], minlength=P)
    n0 = np.bincount(pc[b], minlength=P).astype(float); k0 = np.bincount(pc[b], weights=f[b], minlength=P)
    both = (n1 >= MIN_STRATUM) & (n0 >= MIN_STRATUM); w = n1[both] * n0[both] / (n1[both] + n0[both])
    mh = 100 * float((w * (k1[both] / n1[both] - k0[both] / n0[both])).sum() / w.sum()) if both.any() else np.nan
    # expected from prompt mix alone: the rate of everyone ELSE (all evaluation humans not in this group) on the same prompt, weighted by the group's prompt mix
    no, ko = OTHERS[attr][g]
    prate = np.divide(ko, no, out=np.zeros(P), where=no >= MIN_STRATUM)
    ok = (n1 > 0) & (no >= MIN_STRATUM)
    exp_ = float((n1[ok] * prate[ok]).sum() / n1[ok].sum()) if ok.any() else np.nan
    return 100 * k1.sum() / max(n1.sum(), 1), 100 * k0.sum() / max(n0.sum(), 1), mh, 100 * (float(k1[ok].sum() / max(n1[ok].sum(), 1)) - exp_), 100 * exp_, int(n1[both].sum())

PCODES = {p: i for i, p in enumerate(sorted(ev['prompt_name'].unique()))}
OTHERS = {}
for _a, _gs in [('gender', ['F', 'M']), ('grade_level', ['6', '8', '9', '10', '11', '12']), ('race_ethnicity', list(ev['race_ethnicity'].unique())), ('economically_disadvantaged', list(ev['economically_disadvantaged'].unique())),
                ('student_disability_status', list(ev['student_disability_status'].unique())), ('ell_status', ['Yes', 'No'])]:
    OTHERS[_a] = {}
    for _g in _gs:
        _o = ev[ev[_a] != _g]; _pc = _o['prompt_name'].map(PCODES).to_numpy()
        OTHERS[_a][_g] = (np.bincount(_pc, minlength=len(PCODES)).astype(float), np.bincount(_pc, weights=_o['flag'].to_numpy().astype(float), minlength=len(PCODES)))

def table(attr, groups, ref=None):
    d = ev[(ev[attr] != 'unknown') & ev[attr].isin(groups)].reset_index()
    ref = ref or d[attr].value_counts().idxmax()
    P = len(PCODES)
    smp = ClusterSampler(d['dup_cluster'].to_numpy()); out = []
    for g in groups:
        if g == ref: continue
        if ref not in groups: pass
        dd = d[d[attr].isin([g, ref])].reset_index(drop=True)
        pc, PP = dd['prompt_name'].map(PCODES).to_numpy(), len(PCODES)
        sm = ClusterSampler(dd['dup_cluster'].to_numpy()); rng = np.random.default_rng(SEED)
        full = np.arange(len(dd)); r1, r0, mh, oe, ex, n_used = stats(dd, attr, g, ref, full, pc, PP)
        boot = np.array([stats(dd, attr, g, ref, sm.draw(rng), pc, PP)[2:4] for _ in range(N_BOOT)])
        n1, k1 = int((dd[attr] == g).sum()), int(dd.loc[dd[attr] == g, 'flag'].sum()); n0, k0 = int((dd[attr] == ref).sum()), int(dd.loc[dd[attr] == ref, 'flag'].sum())
        out.append({'attribute': attr, 'group': g, 'reference': ref, 'n': n1, 'flagged': k1, 'raw rate % [95% Wilson]': wil(k1, n1), 'reference rate %': wil(k0, n0),
                    'raw diff (pts)': round(r1 - r0, 2), 'expected from prompt mix %': round(ex, 2), 'observed - expected (pts)': f'{oe:+.2f} [{np.nanpercentile(boot[:, 1], 2.5):+.2f}, {np.nanpercentile(boot[:, 1], 97.5):+.2f}]',
                    'prompt-adjusted diff (pts) [95% boot]': f'{mh:+.2f} [{np.nanpercentile(boot[:, 0], 2.5):+.2f}, {np.nanpercentile(boot[:, 0], 97.5):+.2f}]',
                    'mh_diff': mh, 'usable essays': n_used, 'mh_lo': np.nanpercentile(boot[:, 0], 2.5), 'mh_hi': np.nanpercentile(boot[:, 0], 97.5), 'share of group in prompts with >=20 of both groups %': round(100 * n_used / n1, 1)})
    return out

spec = [('gender', ['F', 'M']), ('grade_level', ['9', '6', '8', '10', '11', '12']), ('race_ethnicity', ['White', 'American Indian/Alaskan Native', 'Asian/Pacific Islander', 'Black/African American', 'Hispanic/Latino', 'Two or more races/Other']),
        ('economically_disadvantaged', ['Not economically disadvantaged', 'Economically disadvantaged']), ('student_disability_status', ['Not identified as having disability', 'Identified as having disability']), ('ell_status', ['No', 'Yes'])]
res = pd.DataFrame([r for a, g in spec for r in table(a, g)] + [dict(r, group=r['group'] + ' (vs grade 10, added after seeing the grade table)') for r in table('grade_level', ['10', '11'], ref='10')])
res.drop(columns=['mh_diff', 'mh_lo', 'mh_hi']).set_index(['attribute', 'group'])

/home/jaiveer-minhas/anaconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


/home/jaiveer-minhas/anaconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


/home/jaiveer-minhas/anaconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


/home/jaiveer-minhas/anaconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


/home/jaiveer-minhas/anaconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


reference     n  flagged raw rate % [95% Wilson]   reference rate %  raw diff (pts)  expected from prompt mix %  \
attribute                  group                                                                                                                                                                                             
gender                     M                                                                                       F  3194       11       0.34 [0.19, 0.62]  0.65 [0.43, 0.98]           -0.30                        0.68   
grade_level                6                                                                                       9   134        0       0.00 [0.00, 2.79]  0.12 [0.04, 0.34]           -0.12                         NaN   
                           8                                                                                       9   976        1       0.10 [0.02, 0.58]  0.12 [0.04, 0.34]           -0.01                         NaN   
                           10                                                                                      9  1059        6       0.57 [0.26, 1.23]  0.12 [0.04, 0.34]            0.45                        1.42   
                           11                                                                                      9  1675       23       1.37 [0.92, 2.05]  0.12 [0.04, 0.34]            1.26                        2.04   
                           12                                                                                      9    47        0       0.00 [0.00, 7.56]  0.12 [0.04, 0.34]           -0.12                        0.20   
race_ethnicity             American Indian/Alaskan Native                                                      White    36        0       0.00 [0.00, 9.64]  0.53 [0.33, 0.86]           -0.53                        0.28   
                           Asian/Pacific Islander                                                              White   471        9       1.91 [1.01, 3.59]  0.53 [0.33, 0.86]            1.38                        0.71   
                           Black/African American                                                              White  1219        4       0.33 [0.13, 0.84]  0.53 [0.33, 0.86]           -0.20                        0.66   
                           Hispanic/Latino                                                                     White  1614        4       0.25 [0.10, 0.64]  0.53 [0.33, 0.86]           -0.28                        0.44   
                           Two or more races/Other                                                             White   236        0       0.00 [0.00, 1.60]  0.53 [0.33, 0.86]           -0.53                        0.67   
economically_disadvantaged Economically disadvantaged                                 Not economically disadvantaged  1455        5       0.34 [0.15, 0.80]  1.09 [0.73, 1.61]           -0.74                        0.81   
student_disability_status  Identified as having disability                       Not identified as having disability   383        1       0.26 [0.05, 1.46]  0.85 [0.59, 1.23]           -0.59                        0.67   
ell_status                 Yes                                                                                    No   340        0       0.00 [0.00, 1.12]  0.54 [0.38, 0.76]           -0.54                        0.30   
grade_level                11 (vs grade 10, added after seeing the grade table)                                   10  1675       23       1.37 [0.92, 2.05]  0.57 [0.26, 1.23]            0.81                        2.04   

                                                                                observed - expected (pts) prompt-adjusted diff (pts) [95% boot]  usable essays  share of group in prompts with >=20 of both groups %  
attribute                  group                                                                                     

**Step 3 · the grade-11 example in numbers.** Grade 11 is flagged 1.37% against 0.12% for grade 9. How much of that is prompt mix? (The expected-from-prompt-mix column above uses the rate of everyone else on the same prompt, so it is not circular.)

In [4]:
g = ev[ev['grade_level'].isin(['9', '11'])]
mix = g.groupby(['grade_level', 'prompt_name']).size().unstack(0).fillna(0).astype(int)
mix['Summer projects share, grade 9 %'] = None
print('share of each grade\'s essays that are Summer projects:')
print((g.assign(sp=g['prompt_name'] == 'Summer projects').groupby('grade_level')['sp'].mean() * 100).round(1).to_string())
print('\nflag rate by grade within Summer projects only:')
sp = g[g['prompt_name'] == 'Summer projects'].groupby('grade_level')['flag'].agg(['size', 'sum', 'mean']); sp['mean'] = (100 * sp['mean']).round(2); print(sp.to_string())
print('\nflag rate by grade outside Summer projects:')
nsp = g[g['prompt_name'] != 'Summer projects'].groupby('grade_level')['flag'].agg(['size', 'sum', 'mean']); nsp['mean'] = (100 * nsp['mean']).round(2); print(nsp.to_string())

share of each grade's essays that are Summer projects:
grade_level
11    91.2
9      0.3

flag rate by grade within Summer projects only:
             size  sum  mean
grade_level                 
11           1527   23  1.51
9               8    0  0.00

flag rate by grade outside Summer projects:
             size  sum  mean
grade_level                 
11            148    0  0.00
9            2574    3  0.12


In [5]:
res.to_csv('../results/21b_fairness_adjusted.csv', index=False)
res = res.replace([np.inf], np.nan); print(len(res), 'comparisons written')
print('intervals excluding zero (prompt-adjusted):'); print(res[(res['mh_lo'] > 0) | (res['mh_hi'] < 0)][['attribute', 'group', 'prompt-adjusted diff (pts) [95% boot]']].to_string(index=False))

15 comparisons written
intervals excluding zero (prompt-adjusted):
                 attribute                      group prompt-adjusted diff (pts) [95% boot]
                    gender                          M                  -0.34 [-0.68, -0.01]
            race_ethnicity    Two or more races/Other                  -1.03 [-1.64, -0.54]
economically_disadvantaged Economically disadvantaged                  -0.53 [-1.01, -0.03]
                ell_status                        Yes                  -0.40 [-0.58, -0.25]


## Interpretation

*(Written after the results were committed, `e6c9481`.)*

| Attribute (group vs reference) | Raw difference (pts) | Prompt-adjusted (pts) [95%] | What explains it |
|---|---|---|---|
| Grade 11 vs 9 | +1.26 | **not estimable** | 91% of grade-11 essays are Summer projects, 0.3% of grade-9 essays; only 8 grade-9 essays share a prompt with grade 11 |
| Grade 11 vs 10 (added after seeing the table) | +0.81 | -0.89 [-3.24, +1.04] | they share Summer projects (1,527 vs 209 essays); no grade effect there. Everyone else on Summer projects is flagged at 2.04%, grade 11 at 1.37% |
| Grade 6, 8, 10, 12 vs 9 | -0.12 to +0.45 | not estimable | no prompt with at least 20 essays of both |
| Asian/Pacific Islander vs White | +1.38 | **+1.16 [-0.30, +2.99]** | prompt mix expected 0.71%, observed 1.91% ([+0.11, +2.64] above expected): **not explained by prompt**; only 9 flagged essays, so unresolved |
| Economically disadvantaged vs not | -0.74 | **-0.53 [-1.01, -0.03]** | about a quarter is prompt mix; most of the gap stays and the interval just excludes zero |
| Girls vs boys (M minus F) | -0.30 | **-0.34 [-0.68, -0.01]** | prompt mix explains none of it; interval touches zero |
| Disability identified vs not | -0.59 | -0.50 [-1.09, +0.29] | includes zero |
| Black, Hispanic vs White | -0.20, -0.28 | -0.35 [-0.84, +0.06], -0.22 [-0.56, +0.13] | include zero |
| Two or more races vs White | -0.53 | -1.03 [-1.64, -0.54] | 0 of 236 flagged; the bootstrap is degenerate with zero events, read the Wilson interval (upper bound 1.60%) |
| English learners vs not | -0.54 | -0.40 | 0 of 340; Wilson upper bound 1.12%, bootstrap degenerate |

**Corrects notebook 21.** There, grade 11 was reported as "prompt mix, adjusted -0.73 [-2.76, +0.89]" against grades 8-10 pooled. That was right in direction but the adjustment against grade 9 alone is impossible, and the only usable comparison (grade 11 vs 10, one prompt) has an interval from -3.2 to +1.0. Better stated: **the grade gap cannot be separated from prompt, and where it can be checked there is no sign of a grade effect.**

**What this shows and does not show.**
- Two of the raw gaps shrink or vanish when prompt is accounted for (grade; part of economic disadvantage). Three do not: economic disadvantage, gender and Asian/Pacific Islander students keep the same sign. Only the first two have intervals that exclude zero, narrowly, and this is 14 comparisons (about 1 in 20 would exclude zero by chance), several confounded with each other. The Asian/Pacific Islander gap is the largest point estimate and the least certain.
- The groups flagged more (not economically disadvantaged, girls, Asian/Pacific Islander students) are the groups that tend to score higher on writing quality in PERSUADE, and the groups flagged less include English learners. This fits the earlier finding (`16b`) that false flags fall on more polished, predictable writing; **writing quality was not measured here, so this is a pattern, not a test.**
- All counts are small: 33 flagged essays in total. No group-level claim here rests on more than 23 flagged essays and most rest on fewer than 10.
- Nothing is a statement about any individual.